In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load People dataset

file_path = "sample_-_superstore.xls"
people_df = pd.read_excel(
    file_path,
    sheet_name="People",
    engine="calamine"
)
people_df.head()

,Regional Manager,Region
0,Sadie Pawthorne,West
1,Chuck Magee,East
2,Roxanne Rodriguez,Central
3,Fred Suzuki,South


In [4]:
people_df.shape

(4, 2)

In [5]:
people_df.info()
print("\nMissing Values:")
print(people_df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Regional Manager  4 non-null      str  
 1   Region            4 non-null      str  
dtypes: str(2)
memory usage: 196.0 bytes

Missing Values:
Regional Manager    0
Region              0
dtype: int64


In [6]:
people_df["Region"].duplicated().sum()

np.int64(0)

In [7]:
# Load Orders dataset

orders_df = pd.read_excel(
    file_path,
    sheet_name="Orders",
    engine="calamine"
)
orders_us = orders_df[
    orders_df["Country/Region"] == "United States"
].copy()
orders_us.shape

(9994, 21)

In [8]:
# Merge orders with People

orders_people = pd.merge(
    orders_us,
    people_df,
    on="Region",
    how="left",
    validate="many_to_one"
)
orders_people[
    ["Order ID", "Region", "Regional Manager"]
].head()

,Order ID,Region,Regional Manager
0,US-2023-103800,Central,Roxanne Rodriguez
1,US-2023-112326,Central,Roxanne Rodriguez
2,US-2023-112326,Central,Roxanne Rodriguez
3,US-2023-112326,Central,Roxanne Rodriguez
4,US-2023-141817,East,Chuck Magee


In [ ]:
# Verify merge results

print("Original Orders Rows:", len(orders_us))
print("Merged Dataset Rows:", len(orders_people))
print(
    "Missing Regional Managers:",
    orders_people["Regional Manager"].isna().sum()
)

Original Orders Rows: 9994
Merged Dataset Rows: 9994
Missing Regional Managers: 0


In [10]:
# Load Returns dataset

returns_df = pd.read_excel(
    file_path,
    sheet_name="Returns",
    engine="calamine"
)
returns_df.head()

,Order ID,Returned
0,US-2023-100762,Yes
1,US-2023-100867,Yes
2,US-2023-102652,Yes
3,US-2023-103373,Yes
4,US-2023-103744,Yes


In [11]:
# Merge Orders, People and Returns

orders_integrated = pd.merge(
    orders_people,
    returns_df,
    on="Order ID",
    how="left",
    validate="many_to_one"
)
orders_integrated.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Regional Manager,Returned
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512,Roxanne Rodriguez,NaN
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870,Roxanne Rodriguez,NaN
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717,Roxanne Rodriguez,NaN
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,Roxanne Rodriguez,NaN
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840,Chuck Magee,NaN


In [ ]:
# Create return flag

orders_integrated["Is_Returned_Order"] = (
    orders_integrated["Returned"] == "Yes"
)
orders_integrated["Is_Returned_Order"].value_counts()

Is_Returned_Order
False    9194
True      800
Name: count, dtype: int64

In [13]:
# Final Data Integration Validation

print("=== DATA INTEGRATION VALIDATION ===")
print("Original Order Rows:", len(orders_us))
print("Integrated Dataset Rows:", len(orders_integrated))
print("Missing Regional Managers:",
      orders_integrated["Regional Manager"].isna().sum()
)
print("Total Unique Orders:", orders_integrated["Order ID"].nunique())
print("Countries:", orders_integrated["Country/Region"].nunique())

=== DATA INTEGRATION VALIDATION ===
Original Order Rows: 9994
Integrated Dataset Rows: 9994
Missing Regional Managers: 0
Total Unique Orders: 5009
Countries: 1


In [14]:
# Final checks

print(
    "Unique Returned Orders:",
    orders_integrated.loc[
        orders_integrated["Is_Returned_Order"],
        "Order ID"
    ].nunique()
)
print("Countries:", orders_integrated["Country/Region"].unique())

Unique Returned Orders: 296
Countries: <StringArray>
['United States']
Length: 1, dtype: str


In [15]:
# Import path

from pathlib import Path
output_folder = Path("data/processed")
output_folder.mkdir(
    parents=True,
    exist_ok=True
)
print("Output folder created:", output_folder)

Output folder created: data\processed


In [16]:
# Prepare US Returns dataset

returns_us = returns_df[
    returns_df["Order ID"].isin(orders_us["Order ID"])
].copy()
print("Orders:", orders_us.shape)
print("Returns:", returns_us.shape)
print("People:", people_df.shape)

Orders: (9994, 21)
Returns: (296, 2)
People: (4, 2)


In [17]:
# Export dataset to CSV

orders_us.to_csv(
    output_folder / "orders_clean.csv",
    index=False,
    encoding="utf-8-sig"
)
returns_us.to_csv(
    output_folder / "returns_clean.csv",
    index=False,
    encoding="utf-8-sig"
)
people_df.to_csv(
    output_folder / "people_clean.csv",
    index=False,
    encoding="utf-8-sig"
)
print("All dataset exported successfully!")

All dataset exported successfully!


In [19]:
# Compare financial totals with SQL Server

print("=== PYTHON FINANCIAL VALIDATION ===")

print(f"Total Rows: {len(orders_us):,}")

print(f"Total Sales: ${orders_us['Sales'].sum():,.2f}")

print(f"Total Profit: ${orders_us['Profit'].sum():,.2f}")

=== PYTHON FINANCIAL VALIDATION ===
Total Rows: 9,994
Total Sales: $2,297,200.86
Total Profit: $286,397.02


## Data Integration & Validation

### Objective

Integrate the Orders, Returns and People datasets to validate relationships between sales transactions, returned orders and regional managers.

### Process

- Import and validate the source datasets.
- Merge Orders with People and Returns.
- Verify data consistency and record counts.
- Export processed datasets for SQL Server.
- Validate financial totals against SQL results.